# Phase 2 — Notebook 01: Document Parsing

**Goal:** Parse PDF, HTML, and DOCX documents. Extract text blocks, images, and tables. Inspect the raw structure before chunking.

**Packages used:** `pymupdf` (fitz), `python-docx`, `pillow`, `pandas`

## 1. Setup

In [1]:
import sys
from pathlib import Path

project_root = Path().resolve().parents[1]
sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env")
print(f"Project root: {project_root}")

Project root: C:\Users\siman\Documents\AI_Repo\Multimodal_Agentic_RAG


## 2. PDF Parsing with PyMuPDF

> Place a sample PDF in `data/` and update the path below.

In [2]:
import fitz  # PyMuPDF

PDF_PATH = project_root / "data" / "Attention.pdf"   # update to your file

if not PDF_PATH.exists():
    print(f"Place a PDF at: {PDF_PATH}")
else:
    doc = fitz.open(str(PDF_PATH))
    print(f"Document      : {PDF_PATH.name}")
    print(f"Pages         : {len(doc)}")
    print(f"Metadata      : {doc.metadata}")

Document      : Attention.pdf
Pages         : 15
Metadata      : {'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'LaTeX with hyperref', 'producer': 'pdfTeX-1.40.25', 'creationDate': 'D:20240410211143Z', 'modDate': 'D:20240410211143Z', 'trapped': '', 'encryption': None}


In [3]:
# Extract text blocks from all pages
text_blocks = []

for page_num, page in enumerate(doc):
    blocks = page.get_text("blocks")   # returns list of (x0, y0, x1, y1, text, block_no, block_type)
    for block in blocks:
        text = block[4].strip()
        block_type = block[6]          # 0=text, 1=image
        if block_type == 0 and text:
            text_blocks.append({
                "page": page_num + 1,
                "text": text,
                "char_count": len(text),
            })

print(f"Total text blocks: {len(text_blocks)}")
for b in text_blocks[:5]:
    print(f"\n  [Page {b['page']}] ({b['char_count']} chars)")
    print(f"  {b['text'][:120]}...")

Total text blocks: 484

  [Page 1] (173 chars)
  Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this pap...

  [Page 1] (25 chars)
  Attention Is All You Need...

  [Page 1] (48 chars)
  Ashish Vaswani∗
Google Brain
avaswani@google.com...

  [Page 1] (42 chars)
  Noam Shazeer∗
Google Brain
noam@google.com...

  [Page 1] (45 chars)
  Niki Parmar∗
Google Research
nikip@google.com...


In [4]:
import pandas as pd

df_blocks = pd.DataFrame(text_blocks)
print("Text block stats:")
print(df_blocks["char_count"].describe())

Text block stats:
count     484.000000
mean       80.607438
std       154.458698
min         1.000000
25%         3.000000
50%         6.000000
75%        89.250000
max      1140.000000
Name: char_count, dtype: float64


In [5]:
# Extract images
image_list = []

for page_num, page in enumerate(doc):
    images = page.get_images(full=True)
    for img in images:
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_list.append({
            "page": page_num + 1,
            "xref": xref,
            "format": base_image["ext"],
            "width": base_image["width"],
            "height": base_image["height"],
            "size_bytes": len(base_image["image"]),
        })

print(f"Total images: {len(image_list)}")
for img in image_list[:5]:
    print(f"  [Page {img['page']}] {img['width']}x{img['height']} {img['format'].upper()} ({img['size_bytes']} bytes)")

Total images: 3
  [Page 3] 1520x2239 PNG (121244 bytes)
  [Page 4] 445x884 PNG (19215 bytes)
  [Page 4] 835x1282 PNG (43491 bytes)


In [8]:
# Extract tables using PyMuPDF table detection
table_list = []

for page_num, page in enumerate(doc):
    tabs = page.find_tables()
    for tab in tabs:
        df = tab.to_pandas()
        try:
            table_repr = df.to_markdown(index=False)   # requires tabulate: uv add tabulate
        except ImportError:
            table_repr = df.to_string(index=False)     # fallback — no extra dependency
        table_list.append({
            "page": page_num + 1,
            "rows": df.shape[0],
            "cols": df.shape[1],
            "markdown": table_repr,
        })

print(f"Total tables: {len(table_list)}")
for t in table_list[:3]:
    print(f"\n  [Page {t['page']}] {t['rows']} rows x {t['cols']} cols")
    print(t['markdown'][:300])

Total tables: 6

  [Page 9] 7 rows x 3 cols
| Col0   | train                                     | PPL BLEU params    |
|        | N d d h d d P ϵ                           | (dev) (dev) ×106   |
|        | model ff k v drop ls steps                |                    |
|:-------|:------------------------------------------|:-----------------

  [Page 10] 5 rows x 3 cols
| Parser                              | Training                 | WSJ 23 F1   |
|:------------------------------------|:-------------------------|:------------|
| Vinyals & Kaiser el al. (2014) [37] | WSJ only, discriminative | 88.3        |
| Petrov et al. (2006) [29]           | WSJ only, discrim

  [Page 13] 8 rows x 39 cols
| Col0   | Col1   | Col2   | Col3   | Col4   | Col5   | Col6   | Col7   | Col8   | Col9   | Col10   | Col11   | Col12   | Col13   | Col14   | Col15   | Col16   | Col17   | Col18   | Col19   | Col20   | Col21   |   Col22 | Col23   | Col24   | Col25   | Col26   | Col27   | Col28   | Col29   | Col

In [9]:
# Extract headings (larger font size = likely heading)
headings = []

for page_num, page in enumerate(doc):
    blocks = page.get_text("dict")["blocks"]
    for block in blocks:
        if "lines" not in block:
            continue
        for line in block["lines"]:
            for span in line["spans"]:
                if span["size"] >= 14 and span["text"].strip():
                    headings.append({
                        "page": page_num + 1,
                        "text": span["text"].strip(),
                        "font_size": round(span["size"], 1),
                        "bold": "Bold" in span.get("font", ""),
                    })

print(f"Candidate headings ({len(headings)}):")
for h in headings[:10]:
    print(f"  [Page {h['page']}] {h['font_size']}pt {'(bold)' if h['bold'] else ''}  {h['text'][:80]}")

Candidate headings (2):
  [Page 1] 17.2pt   Attention Is All You Need
  [Page 1] 20.0pt   arXiv:1706.03762v7  [cs.CL]  2 Aug 2023


## 3. HTML Parsing

In [10]:
# Minimal HTML parsing — no extra library needed (Python stdlib)
from html.parser import HTMLParser

class TextExtractor(HTMLParser):
    SKIP_TAGS = {"script", "style", "head", "meta", "link"}

    def __init__(self):
        super().__init__()
        self.blocks = []
        self._current_tag = None
        self._skip = False

    def handle_starttag(self, tag, attrs):
        self._current_tag = tag
        self._skip = tag in self.SKIP_TAGS

    def handle_data(self, data):
        text = data.strip()
        if not self._skip and text:
            self.blocks.append({"tag": self._current_tag, "text": text})

# Test with a sample HTML string
sample_html = """
<html><body>
  <h1>Introduction</h1>
  <p>This is a sample document for testing HTML parsing.</p>
  <h2>Section 1</h2>
  <p>Paragraph content goes here with more detail.</p>
  <table><tr><td>Cell 1</td><td>Cell 2</td></tr></table>
</body></html>
"""

parser = TextExtractor()
parser.feed(sample_html)
print("Extracted HTML blocks:")
for b in parser.blocks:
    print(f"  <{b['tag']}> {b['text'][:80]}")

Extracted HTML blocks:
  <h1> Introduction
  <p> This is a sample document for testing HTML parsing.
  <h2> Section 1
  <p> Paragraph content goes here with more detail.
  <td> Cell 1
  <td> Cell 2


## 4. DOCX Parsing

In [11]:
# from docx import Document as DocxDocument

# DOCX_PATH = project_root / "data" / "sample.docx"   # update to your file

# if not DOCX_PATH.exists():
#     print(f"Place a DOCX at: {DOCX_PATH}")
# else:
#     docx = DocxDocument(str(DOCX_PATH))

#     print(f"Paragraphs: {len(docx.paragraphs)}")
#     print(f"Tables    : {len(docx.tables)}")
#     print()

#     for i, para in enumerate(docx.paragraphs[:10]):
#         style = para.style.name
#         text  = para.text.strip()
#         if text:
#             print(f"  [{style}] {text[:100]}")

## 5. Parsed Output Summary

In [12]:
print("=" * 50)
print("PARSE SUMMARY")
print("=" * 50)
print(f"Text blocks : {len(text_blocks)}")
print(f"Images      : {len(image_list)}")
print(f"Tables      : {len(table_list)}")
print(f"Headings    : {len(headings)}")
print()
print("→ Inputs ready for chunking (Notebook 02)")

PARSE SUMMARY
Text blocks : 484
Images      : 3
Tables      : 6
Headings    : 2

→ Inputs ready for chunking (Notebook 02)
